# Tutoriel -- pipeline "critical materials"

Ce notebook explique :
1. Comment fonctionnent les 3 pipelines qui génèrent les fichiers `.dat` (à partir des fichiers Excel)
2. Comment utiliser `run_pathway(..., materials=True, ...)` -- tous les paramètres qu'on a définis, et ce qu'ils font
3. Comment lire les résultats et régénérer le dashboard

Tout se lance depuis `projects/critical_materials/` (le dossier de ce notebook).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, 'src')
sys.path.insert(0, '../..')  # repo root, for `shared.utils`

# ADDED BY PAOLO (to validate) -- run_pathway_materials() has been folded into
# shared.utils.run_pathway() behind a `materials=True` flag, to avoid keeping
# two near-duplicate pathway runners in sync by hand.
from shared.utils import run_pathway
from Plot_functions import build_materials_dashboard, build_scenario_selector
import pandas as pd

## 1. Les pipelines -- comment sont générés les `.dat`

Rien n'est écrit à la main dans `ampl_files/*.dat` : ce sont des fichiers **auto-générés** par des scripts Python
qui lisent les fichiers Excel dans `excel_files/`. Si tu modifies un Excel, il faut **relancer le pipeline
correspondant** pour que le changement se propage dans le `.dat`, puis relancer `run_pathway(materials=True, ...)`.

| Pipeline | Source Excel | Sortie | Script |
|---|---|---|---|
| `mi_pipeline` | `Material_intensities_energyscope.xlsx` | `Material_intensity.dat` (+ `technologies_mi_all_years.xlsx`, audit) | `run_build_mi.py` |
| `rr_pipeline` | `Recycling_rates.xlsx` | `Material_recycling.dat` | `run_build_rr.py` |
| `rt_pipeline` | `Recycling_rates.xlsx` | `Material_recycling_process.dat` | `run_build_rt.py` |

`Material_intensity.dat` et `Material_recycling.dat` (les 2 premiers) sont ceux qu'on utilise activement
(approche `recycling_materials`). Le 3e (`rt_pipeline`, approche `recycling_materials_technologies` --
technologies de recyclage en compétition, PV c-Si + EV Battery/Chassis/Motor) est maintenant à jour et
utilisable via `materials_recycling_process=True`.

### 1.1 `mi_pipeline` -- intensités matérielles (`material_intensity`)

Lit `Mapping` (quelle techno EnergyScope correspond à quelle sous-techno de la littérature) +
`MI_Energy`/`MI_Vehicles`/`MI_Vehicles_Bieuville_Clean`/`MI_Vehicles_Public`/`MI_H2` (les valeurs elles-mêmes).

**Astuce vitesse** : générer `technologies_mi_all_years.xlsx` (le fichier d'audit, avec le détail
`Vehicle_Calc_Detail` de chaque conversion g/véhicule -> t/(pkm/h)) est **beaucoup plus lent** que générer
juste le `.dat` (~5 min contre quelques secondes, car il réécrit ~200k lignes avec mise en forme). Si tu
n'as pas besoin de l'audit (juste besoin du `.dat` à jour pour lancer un run), passe `write_xlsx=False`.

In [ ]:
from run_build_mi import main as build_mi

# Génère Material_intensity.dat + technologies_mi_all_years.xlsx (lent, ~5 min)
build_mi()

# Rapide : juste le .dat, sans régénérer l'xlsx d'audit
# build_mi(write_xlsx=False)

# Autres options :
# build_mi(vehicle_source='bieuville')   # source alternative pour les intensités véhicules
# build_mi(scenario='optimiste')          # applique les overrides du scénario 'optimiste' (feuille Overrides)

### 1.2 `rr_pipeline` -- taux de recyclage (`recycling_rate`), approche `recycling_materials`

Lit `Mapping` + `RR_Energy`/`RR_Vehicles`/`RR_Vehicles_Public`/`RR_H2` (taux spécifiques par techno) +
`RR_Global` (taux global par matériau, littérature Graedel et al. 2022 -- 1re des 3 colonnes) +
`Recycling_objective` (cible pour `follow_objective=True`, voir plus bas).

**Logique cellule par cellule** (important) : pour chaque (techno, matériau), le taux **spécifique**
(RR_Energy/RR_Vehicles/...) est utilisé s'il existe ; sinon on retombe sur le taux **global** de
`RR_Global` pour ce matériau. Ça s'applique aussi bien aux technos sans mapping du tout (ex: nucléaire,
hydro) qu'aux technos mappées mais incomplètes (ex: l'éolien n'a qu'1 matériau sur 41 avec un vrai taux
spécifique -- tout le reste retombe sur `RR_Global`).

In [ ]:
from run_build_rr import main as build_rr

build_rr()

# build_rr(scenario='optimiste')   # applique les overrides du scénario 'optimiste'
# build_rr(write_dat=False)         # juste le rapport de couverture, sans écrire le .dat

### 1.3 `rt_pipeline` -- technologies de recyclage en compétition, approche `recycling_materials_technologies`

`python run_build_rt.py` régénère `Material_recycling_process.dat`. `Constraints_recycling_technologies.mod`
est resynchronisé avec `Constraints.mod` -- `materials_recycling_process=True` fonctionne (PV c-Si module/
infrastructure + EV Battery/Chassis/Motor, voir `excel_files/Recycling_rates.xlsx`).

## 2. `run_pathway(..., materials=True, ...)` -- tous les paramètres

Lance une résolution du modèle pathway + contraintes matières, et retourne un dict de résultats
(DataFrames pandas). Sauvegarde aussi les résultats dans `out/<case_study>/` et génère un dashboard
HTML par défaut.

**ADDED BY PAOLO (to validate)** -- `run_pathway_materials()` (l'ancien module séparé) a été replié dans
`shared.utils.run_pathway()` derrière un flag `materials=True`, pour n'avoir qu'un seul endroit à maintenir
(les deux fonctions dérivaient au fil des changements sur `main`, ce qui a causé plusieurs bugs ce soir).
Sans `materials=True`, `run_pathway` se comporte exactement comme avant -- rien ne change pour les runs
qui n'ont pas besoin des matériaux.

| Paramètre | Défaut | Rôle |
|---|---|---|
| `case_study` | *(obligatoire)* | Nom du run -- sert de nom de dossier `out/<case_study>/` |
| `materials` | `False` | **Le nouveau flag** -- `True` charge `Constraints.mod`/`Material_intensity.dat` et active tous les paramètres `materials_*`/`follow_objective*` ci-dessous |
| `N_year_opti` | `30` | Durée de la fenêtre glissante d'optimisation [années]. 30 = tout l'horizon 2020-2050 en une seule fenêtre |
| `N_year_overlap` | `0` | Chevauchement entre fenêtres consécutives [années] |
| `gwp_budget` | `False` | Budget GWP cumulé sur toute la transition [kt CO2-eq.]. `False` = désactivé |
| `gwp_budget_val` | `1224935.4` | Valeur utilisée si `gwp_budget=True`. Seulement quand `materials=True` |
| `CO2_neutrality_2050` | `True` | Si `True`, force `gwp_limit['YEAR_2050'] = CO2_neutrality_2050_val`. Seulement quand `materials=True` |
| `description` | `''` | Description courte, stockée dans le CSV récapitulatif |
| `save_pkl` | `None` | Sauvegarde `_Results.pkl` (+ `_Materials_Results.pkl` si `materials=True`). `None` = résolu automatiquement (`False` sans `materials`, `True` avec) |
| `skip_if_exists` | `False` | Si `True` et que le pkl existe déjà, recharge depuis le disque au lieu de relancer le solve |
| `verbose` | `False` | Affiche les logs AMPL/Gurobi (utile pour déboguer un solve qui rame) |
| `crossover` | `0` | Option Gurobi (0 = pas de crossover après le barrier method, plus rapide). Seulement quand `materials=True` |
| `materials_limit` | `False` | Charge `Material_limits.dat` (plafonds manuels `limit_material`/`limit_material_year`) |
| `materials_recycling` | `False` | Charge `Material_recycling.dat` (approche `recycling_materials`) -- **sans ça, aucun recyclage** (`recycling_rate` reste à 0 partout) |
| `follow_objective` | `False` | Seulement utile si `materials_recycling=True`. `False` = l'optimiseur recycle librement jusqu'au plafond `recycling_rate` (le moins cher, vu que recycler est quasi gratuit face au coût d'enfouissement 0.01 $/t). `True` = force une **égalité exacte** avec `recycling_objective_share` (feuille `Recycling_objective`), pas juste un minimum |
| `materials_recycling_process` | `False` | Approche `recycling_materials_technologies` (voir §1.3) |
| `build_dashboard` | `True` | Génère `out/<case_study>/materials_graphs/` après le run. Seulement quand `materials=True` |

**Toujours actif quand `materials=True`, peu importe les autres paramètres** : `Material_intensity.dat` (le
fichier généré par `mi_pipeline`) est systématiquement chargé -- c'est lui qui donne `material_intensity`,
la quantité de base (utilisée pour calculer `Material_content_year`, `Decommissioned_material`, etc.).

### 2.1 Exemple -- Approche 1, l'optimiseur décide librement

`follow_objective=False` : l'optimiseur recycle jusqu'au plafond technique (`recycling_rate`), pas plus,
pas moins -- pas de contrainte d'objectif imposée.

In [ ]:
results_free = run_pathway(
    'recycling_materials',
    materials=True,
    materials_recycling=True,
    follow_objective=False,
    description="Approche 1 -- optimiseur libre.",
)

rec = results_free['Recycled_material']['Recycled_material']
rec[rec.abs() > 1e-9].groupby('Materials').sum().sort_values(ascending=False)

### 2.2 Exemple -- Approche 1, égalité forcée avec le scénario

`follow_objective=True` : force `recycling_objective_share` (feuille `Recycling_objective`) exactement,
agrégé sur toutes les technos qui ont un vrai `recycling_rate` pour ce matériau (le pipeline `rr_pipeline`
clippe automatiquement la cible à ce qui est techniquement atteignable, pour éviter une infaisabilité).

In [ ]:
results_objective = run_pathway(
    'recycling_materials_objective',
    materials=True,
    materials_recycling=True,
    follow_objective=True,
    description="Approche 1 -- égalité recycling_objective_share.",
)

rec = results_objective['Recycled_material']['Recycled_material']
rec[rec.abs() > 1e-9].groupby('Materials').sum().sort_values(ascending=False)

### 2.3 Exemple -- recharger un run déjà fait (`skip_if_exists`)

Si `out/<case_study>/_Results.pkl` existe déjà, `skip_if_exists=True` recharge directement depuis le
disque au lieu de relancer tout le solve (~2-6 min économisées) -- pratique pour retravailler le
dashboard ou explorer les résultats sans re-solver.

In [ ]:
results_free = run_pathway(
    'recycling_materials',
    materials=True,
    materials_recycling=True,
    follow_objective=False,
    skip_if_exists=True,
)

## 3. Le dict de résultats

`run_pathway(..., materials=True, ...)` retourne un dict de DataFrames pandas. Les clés les plus utiles
pour les matériaux (en plus de tout ce que `run_pathway` normal retourne déjà -- `F_new`, `F_Mult`,
`Assets`, `TotalCost`, ...) :

| Clé | Description |
|---|---|
| `Material_content_year` | Demande matière annuelle [kt/an], indexé (Years, Technologies, Materials) |
| `Material_content_cumulative` | Cumul de `Material_content_year` sur l'horizon -- la dernière année = le total |
| `Decommissioned_material` | Matière démantelée mécaniquement [kt/an], avant toute décision de recyclage |
| `Recycled_material` | Matière effectivement recyclée [kt/an] |
| `Recycled_material_cumulative` | Cumul de `Recycled_material` |
| `Disposed_material` | Matière enfouie/incinérée [kt/an] = `Decommissioned_material - Recycled_material` |

Chaque DataFrame a une seule colonne (même nom que la clé) et un MultiIndex `(Years, Technologies,
Materials)`.

In [ ]:
results_free['Material_content_year'].head()

## 4. Dashboard

`build_dashboard=True` (par défaut quand `materials=True`) génère automatiquement
`out/<case_study>/materials_graphs/index.html` **et** régénère `out/index.html` (le sélecteur qui liste
tous les runs existants) -- pas besoin de le faire à la main.

Si tu veux juste régénérer le dashboard d'un run déjà fait (sans re-solver), utilise `skip_if_exists=True`
(§2.3) -- le dashboard se reconstruit à partir du pkl rechargé.

In [ ]:
# Régénère juste le sélecteur out/index.html (rarement nécessaire, c'est automatique après chaque run)
build_scenario_selector()

In [ ]:
results = run_pathway(
    'Test_realiste',
    materials=True,
    verbose=True,
    materials_recycling=False,
    materials_recycling_cost=False,   # recycling_cost/primary_material_cost=0, disposal_cost=0.01
    follow_objective_full=False,       # force le recyclage à 100% du plafond atteignable, sans coût
)

In [ ]:
results = run_pathway(
    'Test_realiste',
    materials=True,
    verbose=True,
    materials_recycling=False,
    materials_recycling_cost=False,   # recycling_cost/primary_material_cost=0, disposal_cost=0.01
    follow_objective_full=False,       # force le recyclage à 100% du plafond atteignable, sans coût
)

In [ ]:
results = run_pathway(
    'recycling_materials_no_cost',
    materials=True,
    verbose=True,
    materials_recycling=True,
    materials_recycling_cost=False,   # recycling_cost/primary_material_cost=0, disposal_cost=0.01
    follow_objective_full=True,       # force le recyclage à 100% du plafond atteignable, sans coût
)

In [ ]:
run_pathway(
    'recycling_materials_with_cost',
    materials=True,
    verbose=True,
    materials_recycling=True,
    materials_recycling_cost=True,   # <- nouveau paramètre
    follow_objective=False,
)

In [ ]:
run_pathway(
    'recycling_materials_with_objective_cost',
    materials=True,
    verbose=True,
    materials_recycling=True,
    materials_recycling_cost=True,   # <- nouveau paramètre
    follow_objective=True,
)

## 5. Exemple -- `materials_limit=True`, plafond annuel sur un matériau (Nd)

`materials_limit=True` charge `ampl_files/Material_limits.dat`, qui contient des overrides manuels de
`limit_material` (budget cumulé sur tout l'horizon, [t]) et/ou `limit_material_year[y,mat]` (plafond
**annuel**, pas cumulatif, [t/year] -- remis à zéro chaque année).

Le fichier contient actuellement des limites annuelles sur le néodyme (Nd), à tester sur le scénario
`1_baseline_real_cost` :

| Année | 2025 | 2030 | 2035 | 2040 | 2045 | 2050 |
|---|---|---|---|---|---|---|
| Limite [t/an] | 44.91 | 56.93 | 78 | 98.5 | 134.51 | 170.5 |

**Attention à l'indexation par année** : `limit_material_year['YEAR_2025','Nd']` s'applique à la matière
consommée par les installations construites pendant la période **2020_2025** (pas 2025_2030) -- voir
`Constraints.mod`'s `material_content_year_calc` : `y in PHASE_STOP[p]`, donc `YEAR_2025` = fin de la
phase `2020_2025`.

**`iis_find`** (nouveau paramètre, seulement quand `materials=True`) : si le run est infaisable, Gurobi
calcule automatiquement l'IIS (le sous-ensemble minimal de contraintes en conflit) quand `iis_find=True`
(par défaut) -- **ça peut prendre des heures** sur un modèle de cette taille (on l'a vérifié : >14h sans
terminer). Pour un premier essai, mets `iis_find=False` : ça donne un verdict feasible/infeasible en
~30-60 secondes, sans calculer l'IIS. Repasse à `iis_find=True` seulement si tu veux vraiment le détail
du conflit et que tu es prêt à attendre longtemps.

**Statut actuel (vérifié)** : avec ces limites, `1_baseline_real_cost` est **infaisable**. Des tests
ciblés (forcer `F_new` à 0 pour les technos EV -- CAR_EV/SUV_EV/BUS_EV/etc. -- à partir de 2025_2030)
montrent que ce n'est **pas** la mobilité électrique qui bloque : même sans aucune nouvelle voiture/bus
électrique, ça reste infaisable. La piste la plus probable est la contrainte `elecgen_subtech_fixed_split`
(`PES_scenarios.mod`) qui fige la part des éoliennes à aimant permanent (PMSG, qui consomment du Nd) à
~50% de tout le nouvel éolien onshore -- mais ça reste à confirmer proprement (le test direct en forçant
aussi les PMSG à 0 est confondu avec cette même contrainte de répartition).